# Manual Hybrid K-fold Comparison

This notebook removes the alpha/blending experiment completely.

It compares:

- LSTM/MCLDNN baseline
- existing manual hybrid without k-fold
- new manual hybrid using k-fold trained normal-attention and differential-attention models

Hybrid rule mirrors the earlier non-kfold hybrid:

- for low/transition SNR values, choose normal vs differential attention using validation accuracy
- for cleaner SNR values, use normal attention
- no alpha, no probability mixing, no monotonic trend forcing

Final output folder:

```text
experiments/5class_manual_hybrid_kfold_compare/
```

In [ ]:
# CELL 1: Setup repo and paths
import os
import sys
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

import pandas as pd
from IPython.display import Image, display, FileLink

os.environ['KERAS_BACKEND'] = 'tensorflow'

REPO_URL = 'https://github.com/akshlabh/amr-5-class.git'
WORK_DIR = Path('/kaggle/working/amr-5-class')

DATASET_CANDIDATES = [
    Path('/kaggle/input/datasets/gustavopolicarpo/rml201610a-dict/RML2016.10a_dict.dat'),
    Path('/kaggle/input/rml201610a-dict/RML2016.10a_dict.dat'),
    Path('/kaggle/input/radioml2016-10a/RML2016.10a_dict.pkl'),
    Path('/kaggle/input/radioml2016-10a/RML2016.10a_dict.dat'),
    Path('data/RML2016.10a_5class.pkl'),
    Path('data/RML2016.10a_dict.pkl'),
    Path('data/RML2016.10a_dict.dat'),
]

def find_attached_repo():
    input_root = Path('/kaggle/input')
    if not input_root.exists():
        return None
    for root in input_root.glob('**'):
        if (root / 'src' / 'train.py').exists() and (root / 'configs').exists():
            return root
    return None

if (Path.cwd() / 'src' / 'train.py').exists():
    WORK_DIR = Path.cwd()
    print('Using current repo:', WORK_DIR)
elif (WORK_DIR / 'src' / 'train.py').exists():
    print('Using existing repo:', WORK_DIR)
else:
    attached = find_attached_repo()
    if attached is not None:
        print('Copying attached repo from:', attached)
        if WORK_DIR.exists():
            shutil.rmtree(WORK_DIR)
        shutil.copytree(attached, WORK_DIR)
    else:
        print('Cloning repo from GitHub...')
        subprocess.run(['git', 'clone', REPO_URL, str(WORK_DIR)], check=True)

os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

DATASET = next((p for p in DATASET_CANDIDATES if p.exists()), None)
assert DATASET is not None, 'Dataset not found. Set DATASET manually in this cell.'

BASELINE_DIR = Path('experiments/5class_baseline')
NORMAL_DIR = Path('experiments/5class_attention')
DIFF_DIR = Path('experiments/5class_diffattention')
HYBRID_DIR = Path('experiments/5class_hybrid_snr_aware_attention')
NORMAL_KFOLD_DIR = Path('experiments/5class_attention_kfold/kfold')
DIFF_KFOLD_DIR = Path('experiments/5class_diffattention_kfold/kfold')
OUT_DIR = Path('experiments/5class_manual_hybrid_kfold_compare')

print('Working dir:', Path.cwd())
print('Dataset    :', DATASET)
print('Dataset OK :', DATASET.exists())

In [ ]:
# CELL 2: Check required files
required = [
    'src/train.py',
    'src/train_kfold.py',
    'src/evaluate_hybrid_snr_aware_attention.py',
    'src/evaluate_manual_hybrid_kfold_compare.py',
    'src/models/mcldnn.py',
    'src/models/mcldnn_attention.py',
    'src/models/mcldnn_diffattention.py',
    'configs/exp_5class_baseline.yaml',
    'configs/exp_5class_attention.yaml',
    'configs/exp_5class_diffattention.yaml',
    'configs/exp_5class_attention_kfold.yaml',
    'configs/exp_5class_diffattention_kfold.yaml',
]

for f in required:
    print(('OK      ' if Path(f).exists() else 'MISSING ') + f)
    assert Path(f).exists(), f'Missing required file: {f}'

def kfold_ready(kfold_dir, n_folds=5):
    return all((kfold_dir / f'fold_{i}/best_model.weights.h5').exists() for i in range(n_folds))

print('Baseline acc CSV:', (BASELINE_DIR / 'results/acc_per_snr.csv').exists())
print('Existing non-kfold hybrid CSV:', (HYBRID_DIR / 'results/acc_per_snr.csv').exists())
print('Normal attention kfold ready:', kfold_ready(NORMAL_KFOLD_DIR))
print('Diff attention kfold ready:', kfold_ready(DIFF_KFOLD_DIR))

In [ ]:
# CELL 3: Ensure LSTM baseline and existing non-kfold hybrid results exist
# If your repo already contains these result files, this cell skips training/evaluation.

def run_stream(cmd):
    print('Running:', ' '.join(map(str, cmd)))
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    rc = process.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, process.args)

# LSTM/MCLDNN baseline for comparison
baseline_csv = BASELINE_DIR / 'results/acc_per_snr.csv'
if baseline_csv.exists():
    print('LSTM baseline result found:', baseline_csv)
else:
    print('LSTM baseline result missing; training baseline...')
    run_stream([sys.executable, '-u', 'src/train.py', '--config', 'configs/exp_5class_baseline.yaml', '--datasetpath', str(DATASET)])
    assert baseline_csv.exists(), f'Missing after training: {baseline_csv}'

# Existing non-kfold manual hybrid
normal_weights = NORMAL_DIR / 'checkpoints/best_model.weights.h5'
diff_weights = DIFF_DIR / 'checkpoints/best_model.weights.h5'
for name, weights, cfg in [
    ('normal_attention', normal_weights, 'configs/exp_5class_attention.yaml'),
    ('diff_attention', diff_weights, 'configs/exp_5class_diffattention.yaml'),
]:
    if weights.exists():
        print(f'{name} checkpoint found:', weights)
    else:
        print(f'{name} checkpoint missing; training...')
        run_stream([sys.executable, '-u', 'src/train.py', '--config', cfg, '--datasetpath', str(DATASET)])
        assert weights.exists(), f'Missing after training: {weights}'

nonkfold_csv = HYBRID_DIR / 'results/acc_per_snr.csv'
if nonkfold_csv.exists():
    print('Existing non-kfold hybrid result found:', nonkfold_csv)
else:
    print('Existing non-kfold hybrid result missing; evaluating now...')
    run_stream([
        sys.executable, '-u', 'src/evaluate_hybrid_snr_aware_attention.py',
        '--datasetpath', str(DATASET),
        '--normal-weights', str(normal_weights),
        '--diff-weights', str(diff_weights),
        '--output-dir', str(HYBRID_DIR),
        '--selection-mode', 'validation_best',
        '--low-snr-max', '2',
    ])
    assert nonkfold_csv.exists(), f'Missing after evaluation: {nonkfold_csv}'

In [ ]:
# CELL 4: Train missing k-fold attention models
# This trains normal attention k-fold and diff attention k-fold only if fold checkpoints are missing.

jobs = [
    ('normal_attention_kfold', NORMAL_KFOLD_DIR, 'configs/exp_5class_attention_kfold.yaml'),
    ('diff_attention_kfold', DIFF_KFOLD_DIR, 'configs/exp_5class_diffattention_kfold.yaml'),
]

for name, kfold_dir, cfg in jobs:
    if kfold_ready(kfold_dir):
        print(f'{name}: all fold checkpoints found, skipping training.')
    else:
        print(f'{name}: missing one or more fold checkpoints; training now...')
        run_stream([sys.executable, '-u', 'src/train_kfold.py', '--config', cfg, '--datasetpath', str(DATASET)])
        assert kfold_ready(kfold_dir), f'K-fold training finished but checkpoints are missing: {kfold_dir}'

In [ ]:
# CELL 5: Evaluate manual hybrid using k-fold models and compare with existing results
if OUT_DIR.exists():
    print('Removing old comparison outputs:', OUT_DIR)
    shutil.rmtree(OUT_DIR)

run_stream([
    sys.executable, '-u', 'src/evaluate_manual_hybrid_kfold_compare.py',
    '--datasetpath', str(DATASET),
    '--normal-kfold-dir', str(NORMAL_KFOLD_DIR),
    '--diff-kfold-dir', str(DIFF_KFOLD_DIR),
    '--lstm-baseline-csv', str(BASELINE_DIR / 'results/acc_per_snr.csv'),
    '--nonkfold-hybrid-csv', str(HYBRID_DIR / 'results/acc_per_snr.csv'),
    '--output-dir', str(OUT_DIR),
    '--n-folds', '5',
    '--low-snr-max', '2',
    '--selection-mode', 'validation_best',
])

assert (OUT_DIR / 'results/manual_hybrid_kfold_vs_nonkfold_vs_lstm.csv').exists()
assert (OUT_DIR / 'figures/manual_hybrid_kfold_vs_nonkfold_vs_lstm_acc_vs_snr.png').exists()
print('Manual hybrid k-fold comparison complete.')

In [ ]:
# CELL 6: Display result tables
summary = pd.read_csv(OUT_DIR / 'results/manual_hybrid_kfold_vs_nonkfold_vs_lstm.csv')
fold_rows = pd.read_csv(OUT_DIR / 'results/manual_hybrid_kfold_fold_acc_per_snr.csv')

print('Final comparison: LSTM baseline vs existing non-kfold hybrid vs k-fold hybrid')
display(summary)

print('Per-fold k-fold hybrid values')
display(fold_rows)

In [ ]:
# CELL 7: Display final comparison plots
figs = [
    OUT_DIR / 'figures/manual_hybrid_kfold_vs_nonkfold_vs_lstm_acc_vs_snr.png',
    OUT_DIR / 'figures/manual_hybrid_kfold_delta_vs_comparisons.png',
]

for fig in figs:
    print(fig)
    display(Image(filename=str(fig)))

In [ ]:
# CELL 8: Create repo-ready zip for comparison results only
stamp = datetime.now().strftime('%Y%m%d_%H%M')
zip_base = Path('/kaggle/working') / f'manual_hybrid_kfold_compare_repo_ready_{stamp}'
zip_path = shutil.make_archive(
    str(zip_base),
    'zip',
    root_dir=str(WORK_DIR),
    base_dir='experiments/5class_manual_hybrid_kfold_compare',
)

print('Created repo-ready zip:', zip_path)
print('Extract this at repo root. It will create/update:')
print('  experiments/5class_manual_hybrid_kfold_compare/')
display(FileLink(zip_path))